In [1]:
!rm -rf /kaggle/working/ImgForge
!git clone https://github.com/UtkarshKumarJha/ImgForge.git /kaggle/working/ImgForge
%cd /kaggle/working/ImgForge
!pip install -q imagehash albumentations

Cloning into '/kaggle/working/ImgForge'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 96 (delta 0), reused 1 (delta 0), pack-reused 91 (from 1)
Receiving objects: 100% (96/96), 58.51 MiB | 35.94 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/ImgForge


In [2]:
import os
os.makedirs("/kaggle/working/casia_link", exist_ok=True)
os.symlink(
    "/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2",
    "/kaggle/working/casia_link/CASIA2.0_revised"
)

In [3]:
RAW_DIR = "/kaggle/working/casia_link"
!python data/prepare_dataset.py --raw-dir "{RAW_DIR}" --out-dir data --seed 42
!wc -l data/train.csv data/val.csv data/test.csv

Authentic: 7491  |  Tampered: 5123
Deduplicating by perceptual hash...
  Dedup: removed 803 near-duplicates, kept 11811
Splitting by source group...
  OK: no shared Tp sources across splits
train: 9448 total | authentic=5223 tampered=4225 → data/train.csv
val  : 1181 total | authentic=903 tampered=278 → data/val.csv
test : 1182 total | authentic=927 tampered=255 → data/test.csv
Done — CSV index files created, no files copied.
  9449 data/train.csv
  1182 data/val.csv
  1183 data/test.csv
 11814 total


In [4]:
import os, shutil, csv

# Recreate the experiment log from seed 1 results
# Copy this from the output of your previous session
# Check your previous output tab for the path

# If you saved the previous notebook version, create a dataset from its output
# and attach it, then set PREV below:
PREV = None  # e.g. "/kaggle/input/your-previous-output-dataset/ImgForge"

if PREV:
    for subdir in ["results", "models"]:
        src = os.path.join(PREV, subdir)
        if os.path.isdir(src):
            os.makedirs(subdir, exist_ok=True)
            for f in os.listdir(src):
                shutil.copy2(os.path.join(src, f), os.path.join(subdir, f))
            print(f"Restored {subdir}/")

# Verify
if os.path.exists("results/experiment_log.csv"):
    with open("results/experiment_log.csv") as f:
        n = sum(1 for _ in f) - 1
    print(f"{n} runs already logged")
else:
    print("No previous results — will run all 36")

# Check disk
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")

No previous results — will run all 36
Disk: 20.8 GB free / 21.0 GB total


In [5]:
!python run_sweep.py \
    --data-dir data \
    --num-epochs 50 \
    --batch-size 16 \
    --num-workers 2 \
    --ela-cache-dir data/ela_cache \
    --ela-quality 75

ELA cache: 11811 images, 0 already cached
  Disk free [before warmup]: 19.4 GB
Warming ELA cache: 100%|██████████████████| 11811/11811 [02:02<00:00, 96.56it/s]
ELA cache ready: 11811 files
  Disk free [after warmup]: 14.8 GB

Sweep: 4 archs x 3 modes x 3 seeds = 36 combos
Already completed: 0
Remaining: 36

TRAIN: efficientnet_b0_rgb_s1
  Disk free [before training]: 14.8 GB
[train] Loaded 9448 samples (mode=rgb)
[val] Loaded 1181 samples (mode=rgb)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 140MB/s]
Epoch 1/50 [Val]: 100%|█████████████████████████| 74/74 [00:03<00:00, 22.26it/s]
Epoch 01 | Train Loss: 0.5963 Acc: 0.6739 F1: 0.6731 | Val Loss: 0.5782 Acc: 0.7130 F1: 0.6691
  > New best model saved (val_f1=0.6691)
Epoch 2/50 [Val]: 100%|█████████████████████████| 74/74 [00:03<00:00, 23.15it/s]


In [6]:
import pandas as pd
log = pd.read_csv("results/experiment_log.csv")
print(f"Completed: {len(log)} / 36 runs\n")
print(log[["arch", "input_mode", "seed", "f1_macro", "accuracy"]].to_string(index=False))

Completed: 36 / 36 runs

           arch input_mode  seed  f1_macro  accuracy
efficientnet_b0        rgb     1    0.6401    0.6997
efficientnet_b0        ela     1    0.7932    0.8503
efficientnet_b0    rgb_ela     1    0.6709    0.7327
            mlp        rgb     1    0.5423    0.6548
            mlp        ela     1    0.5242    0.6142
            mlp    rgb_ela     1    0.5528    0.6810
resnet18_bilstm        rgb     1    0.6712    0.7259
resnet18_bilstm        ela     1    0.7789    0.8325
resnet18_bilstm    rgb_ela     1    0.7110    0.7885
      swin_tiny        rgb     1    0.5166    0.6244
      swin_tiny        ela     1    0.4395    0.7843
      swin_tiny    rgb_ela     1    0.5680    0.6320
efficientnet_b0        rgb     2    0.6371    0.7081
efficientnet_b0        ela     2    0.8241    0.8697
efficientnet_b0    rgb_ela     2    0.6771    0.7301
            mlp        rgb     2    0.5335    0.6091
            mlp        ela     2    0.5233    0.6413
            mlp    rg

In [7]:
import os
for d in ["results", "models", "data/ela_cache"]:
    if os.path.isdir(d):
        n = len(os.listdir(d))
        print(f"{d}/  → {n} files")
    else:
        print(f"{d}/  → (not created yet)")

print("\n✓ Now click 'Save Version' → 'Quick Save' in the top-right menu")
print("  to persist everything before the session ends.")

results/  → 1 files
models/  → 38 files
data/ela_cache/  → 11811 files

✓ Now click 'Save Version' → 'Quick Save' in the top-right menu
  to persist everything before the session ends.
